# 4.wav2vect · 1. Extracción de embeddings wav2vec2 (enfoque SOTA)

En lugar de features hechas a mano o de una CNN entrenada desde cero (que sobre 107 sujetos
rinde ≈ azar, como ya vimos), se usa **transfer learning**: `wav2vec2-base`, preentrenado por
Meta en **~960 h de habla**, como **extractor de representaciones**. Es el enfoque del estado
del arte audio-only en DAIC-WOZ (F1 ~0.69–0.79 con este tipo de modelos).

**Diseño:**
- Mismo aislamiento de habla del paciente (transcript) y mismo ventaneo de 4 s que en el resto
  del trabajo → comparabilidad directa y agregación a sesión.
- Por ventana: se pasa el audio por wav2vec2 y se toma la **capa 8** (la que la literatura
  señala como más informativa para depresión), promediando sus frames → **1 vector de 768-dim**.
- Modelo **congelado** (sin fine-tuning): extracción de features. El fine-tuning completo (para
  acercarse a 0.79) requiere GPU y se haría aparte (Colab).

**Salida:** `output/df_wav2vec_features.csv`, una fila por ventana: metadatos + 768 columnas de
embedding. El modelado (notebook 2) lo consume con GroupKFold por sesión y agregación a sujeto.

> Kernel: **unir_w2v (wav2vec2)**. La extracción tarda (~40–60 min en CPU; es una pasada de un
> transformer por cada ventana). Déjalo corriendo.

## 0. Setup

In [2]:
import numpy as np, pandas as pd
import torch, librosa
from transformers import Wav2Vec2Model, AutoFeatureExtractor
from tqdm import tqdm
import warnings, logging
warnings.filterwarnings('ignore'); logging.disable(logging.WARNING)

from pathlib import Path
PROJECT_DIR = Path.home().as_posix() + '/Desktop/Master/UNIR_IA_TFE'   # ruta local del proyecto
path_output = PROJECT_DIR + '/output'

SR          = 16_000
FRAME_LEN   = int(0.025 * SR)
MIN_SEG_DUR = 0.3
WIN_DUR, WIN_HOP, MIN_WIN = 4.0, 4.0, 2.0

MODEL_NAME = 'facebook/wav2vec2-base'
LAYER      = 8      # capa del transformer a extraer (0=embeddings, 1..12=capas)
EMB_DIM    = 768

In [3]:
df_sample = pd.read_csv(path_output + '/df_sample.csv')
print('Sesiones:', len(df_sample))

feat_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
w2v = Wav2Vec2Model.from_pretrained(MODEL_NAME).eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
w2v = w2v.to(device)
print('wav2vec2 cargado en', device)

Sesiones: 186


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 16216.48it/s]

wav2vec2 cargado en cpu


## 1. Aislamiento de habla (reutilizado verbatim) y ventaneo

In [4]:
def load_patient_segments(path_audio: str,
                           path_transcript: str,
                           speaker: str = 'Participant') -> list:
    """
    Carga el audio y extrae únicamente los segmentos de habla verbal del paciente.

    Decisión: aislar el habla del paciente via timestamps del transcript (ground
    truth anotado), en lugar de VAD automático. Esto evita contaminar las
    features con la voz de Ellie o los silencios interturno, que son
    acústicamente distintos a la habla del paciente y distorsionarían
    métricas prosódicas y de calidad vocal.

    Se excluyen marcadores no verbales (<laughter>, <cough>, <synch>, etc.)
    para no contaminar features acústicas con audio que no es habla.

    Se descartan segmentos < MIN_SEG_DUR (0.3 s) para evitar artefactos en
    el cálculo de MFCCs y estimación de F0.
    """
    df_t = pd.read_csv(path_transcript, sep='\t')
    df_t.columns = df_t.columns.str.strip()
    df_t = (df_t[df_t['speaker'] == speaker]
              .sort_values('start_time')
              .reset_index(drop=True))

    # Excluir marcadores no verbales: <laughter>, <cough>, <synch>, etc.
    df_t = df_t[~df_t['value'].str.strip().str.startswith('<', na=False)].reset_index(drop=True)

    y, _ = librosa.load(path_audio, sr=SR, mono=True)

    segments = []
    for _, row in df_t.iterrows():
        dur = row['stop_time'] - row['start_time']
        if dur < MIN_SEG_DUR:
            continue

        s = int(row['start_time'] * SR)
        e = int(row['stop_time']  * SR)
        seg = y[s:e]

        if len(seg) >= FRAME_LEN:
            segments.append(seg)

    return segments


## ─────────────────────────────────────────────────────────────────────────────

In [5]:
def to_windows(segments):
    win, hop, minw = int(WIN_DUR*SR), int(WIN_HOP*SR), int(MIN_WIN*SR)
    out = []
    for seg in segments:
        n = len(seg)
        if n < minw: continue
        s = 0
        while s < n:
            w = seg[s:s+win]
            if len(w) >= minw: out.append(w)
            s += hop
    return out

## 2. Embedding wav2vec2 por ventana (capa 8, mean-pool)

In [6]:
@torch.no_grad()
def window_embedding(w):
    inp = feat_extractor(w.astype(np.float32), sampling_rate=SR, return_tensors='pt')
    out = w2v(inp.input_values.to(device), output_hidden_states=True)
    # capa LAYER, promedio sobre los frames temporales -> (768,)
    return out.hidden_states[LAYER].mean(dim=1).squeeze(0).cpu().numpy()

# sanity check
_seg = load_patient_segments(df_sample.iloc[0]['path_audio'], df_sample.iloc[0]['path_transcript'])
_e = window_embedding(to_windows(_seg)[0])
print('embedding dim:', _e.shape[0])

embedding dim: 768


## 3. Extracción sobre toda la muestra

In [7]:
META = ['participant_id', 'phq8_score', 'phq8_binary', 'gender', 'split']
emb_cols = [f'emb_{i:03d}' for i in range(EMB_DIM)]

rows = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Sesiones"):
    try:
        segs = load_patient_segments(row['path_audio'], row['path_transcript'])
    except Exception as e:
        print(f"  [{row.get('participant_id','?')}] ERROR: {e}"); continue
    for wi, w in enumerate(to_windows(segs)):
        rec = {c: row[c] for c in META}
        rec['window_idx'] = wi
        rec.update(dict(zip(emb_cols, window_embedding(w))))
        rows.append(rec)

df_emb = pd.DataFrame(rows)
print(f"\nVentanas: {len(df_emb)} | sesiones: {df_emb['participant_id'].nunique()} | cols: {df_emb.shape[1]}")
print(f"Ventanas por sesión: media {df_emb.groupby('participant_id').size().mean():.1f}")

Sesiones: 100%|██████████| 186/186 [1:03:54<00:00, 20.62s/it]



Ventanas: 19132 | sesiones: 186 | cols: 774
Ventanas por sesión: media 102.9


## 4. Guardado

In [8]:
out_path = path_output + '/df_wav2vec_features.csv'
df_emb.to_csv(out_path, index=False)
print(f"Guardado: {out_path}  —  shape {df_emb.shape}")

Guardado: C:/Users/tblxa/Desktop/Master/UNIR_IA_TFE/output/df_wav2vec_features.csv  —  shape (19132, 774)
